# 📖 Notebook 4: Data Sovereignty Audit

**Goal**: Build an auditing system that proves where data lives and how it moves between regions — essential for GDPR compliance reports.

## Learning Objectives

By the end of this notebook, you'll understand:
- What data sovereignty means and why auditing matters
- How to generate a data residency report across regions
- How to detect compliance violations (data in the wrong region)
- How to build a GDPR compliance dashboard
- Why Microsoft invests heavily in Azure Purview and Compliance Manager

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/gdpr-paired-regions
docker-compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Compare data across both region databases.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
from datetime import datetime
from tabulate import tabulate

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

REGIONS = {
    "eu-west": EU_WEST_CONFIG,
    "eu-north": EU_NORTH_CONFIG
}

def get_connection(region):
    return psycopg2.connect(**REGIONS[region])

# Verify connections
for region in REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

## 1. What is Data Sovereignty?

**Data sovereignty** means that data is subject to the laws of the country where it is stored. For GDPR:

- Data stored in the **Netherlands** is subject to Dutch + EU law
- Data stored in **Ireland** is subject to Irish + EU law
- Data stored in the **US** is subject to US law (which may conflict with GDPR!)

### Why Auditing Matters

GDPR Article 30 requires you to maintain a **Record of Processing Activities (ROPA)**. This means you must document:

1. **What** personal data you process
2. **Why** you process it (legal basis)
3. **Where** it is stored (geographic location)
4. **Who** has access to it
5. **How long** you keep it
6. **Where** it is transferred (cross-border flows)

A Data Protection Authority (DPA) can ask for this report at any time. If you can't produce it, you're already in violation.

Let's build the tools to generate these reports.

In [ ]:
# ── Audit Tool 1: Data Residency Map ──────────────────────
# Shows where user data is physically stored across regions.

def generate_residency_map():
    """
    Scans all regions and builds a map of where each user's data lives.
    
    In Azure, this is handled by Azure Purview (now Microsoft Purview),
    which automatically discovers and classifies data across services.
    """
    residency_map = {}  # email -> {regions where data exists}

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, email, full_name, country_code, home_region
            FROM users
            ORDER BY id
        """)
        for row in cur.fetchall():
            user_id, email, name, country, home = row
            if email not in residency_map:
                residency_map[email] = {
                    "user_id": user_id,
                    "name": name,
                    "country": country,
                    "home_region": home,
                    "found_in": []
                }
            residency_map[email]["found_in"].append(region)
        conn.close()

    return residency_map


# Generate and display the map
print("🗺️  DATA RESIDENCY MAP")
print("=" * 80)
print("Shows where each user's PII is physically stored.\n")

rmap = generate_residency_map()

table_data = []
for email, info in sorted(rmap.items(), key=lambda x: x[1]["user_id"]):
    regions_str = ", ".join(info["found_in"])
    is_replicated = "Yes" if len(info["found_in"]) > 1 else "No"
    table_data.append([
        info["user_id"],
        info["name"],
        info["country"],
        info["home_region"],
        regions_str,
        is_replicated
    ])

print(tabulate(
    table_data,
    headers=["ID", "Name", "Country", "Home Region", "Data Stored In", "Replicated?"],
    tablefmt="grid"
))

## 2. Compliance Violation Detection

A compliance violation occurs when data is stored in the **wrong region**. For example:

- A Dutch user's data ends up in a non-EU database
- A user's `home_region` doesn't match the database they're actually in
- Data exists in a region that the user's country doesn't allow

Let's build a violation detector:

In [ ]:
# ── Audit Tool 2: Compliance Violation Detector ────────────

# Which countries are allowed in which regions?
REGION_ALLOWED_COUNTRIES = {
    "eu-west": ["NL", "BE", "FR", "DE", "LU", "AT", "CH"],
    "eu-north": ["IE", "SE", "FI", "DK", "NO", "IS"],
}

def detect_violations():
    """
    Scans all regions for data residency violations.
    
    A violation is when a user's data is stored in a region that
    doesn't match their country's assigned region.
    
    Note: Being in a PAIRED region is OK (for DR replication).
    Being in a completely WRONG geography would be a violation.
    """
    violations = []
    warnings = []

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, email, full_name, country_code, home_region
            FROM users
        """)

        for row in cur.fetchall():
            user_id, email, name, country, home_region = row
            allowed = REGION_ALLOWED_COUNTRIES.get(home_region, [])

            if country not in allowed:
                violations.append({
                    "user_id": user_id,
                    "email": email,
                    "name": name,
                    "country": country,
                    "home_region": home_region,
                    "found_in": region,
                    "severity": "HIGH",
                    "reason": f"Country {country} is not assigned to region {home_region}"
                })

            # Check if data is in a non-home region (replication is OK, but flag it)
            if region != home_region:
                # Both eu-west and eu-north are in EU — this is OK for paired regions
                warnings.append({
                    "user_id": user_id,
                    "email": email,
                    "name": name,
                    "home_region": home_region,
                    "found_in": region,
                    "severity": "INFO",
                    "reason": "Data replicated to paired region (within EU — compliant)"
                })

        conn.close()

    return violations, warnings


print("🔍 COMPLIANCE VIOLATION SCAN")
print("=" * 60)

violations, warnings = detect_violations()

if violations:
    print(f"\n🚨 {len(violations)} VIOLATION(S) FOUND:")
    for v in violations:
        print(f"   ❌ [{v['severity']}] User {v['name']} ({v['email']})")
        print(f"      {v['reason']}")
else:
    print("\n✅ No violations found — all data is in the correct region!")

if warnings:
    print(f"\nℹ️  {len(warnings)} informational notice(s):")
    for w in warnings:
        print(f"   📋 User {w['name']}: {w['reason']}")
else:
    print("\nℹ️  No cross-region replication detected.")

In [ ]:
# ── Let's create a deliberate violation to see the detector work ──

print("⚠️  Creating a Deliberate Violation for Demo")
print("=" * 50)
print("Inserting a Swedish user into EU-West (wrong region)...\n")

conn = get_connection("eu-west")
cur = conn.cursor()

# This Swedish user should be in eu-north, not eu-west!
cur.execute("""
    INSERT INTO users (email, full_name, phone, country_code, home_region, consent_given)
    VALUES ('violation.test@example.se', 'Test Violation User', '+46-555-0000', 'SE', 'eu-west', TRUE)
    ON CONFLICT (email) DO NOTHING
""")
conn.commit()
conn.close()

# Run the detector again
print("🔍 Re-running violation scan...\n")
violations, warnings = detect_violations()

if violations:
    print(f"🚨 {len(violations)} VIOLATION(S) FOUND:")
    for v in violations:
        print(f"   ❌ [{v['severity']}] User {v['name']} ({v['email']})")
        print(f"      Country: {v['country']}, Home Region: {v['home_region']}, Found In: {v['found_in']}")
        print(f"      Issue: {v['reason']}")
        print(f"      Fix: Move this user's data to the correct region")

# Clean up the violation
conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("DELETE FROM users WHERE email = 'violation.test@example.se'")
conn.commit()
conn.close()
print("\n🧹 Violation test data cleaned up")

## 3. Consent Audit Report

GDPR requires you to prove that users **gave consent** before you processed their data. Let's generate a consent audit report:

In [ ]:
# ── Audit Tool 3: Consent Status Report ───────────────────

def generate_consent_report():
    """
    Generates a consent status report across all regions.
    
    GDPR requires you to prove:
    - When consent was given
    - What consent was given for (purpose)
    - That consent was freely given (not coerced)
    - That users can withdraw consent at any time
    """
    report = {"total_users": 0, "consented": 0, "no_consent": 0, "by_region": {}}

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()

        # Overall consent stats
        cur.execute("""
            SELECT
                COUNT(*) as total,
                COUNT(*) FILTER (WHERE consent_given = TRUE) as consented,
                COUNT(*) FILTER (WHERE consent_given = FALSE OR consent_given IS NULL) as no_consent
            FROM users
            WHERE home_region = %s
        """, (region,))
        stats = cur.fetchone()

        report["by_region"][region] = {
            "total": stats[0],
            "consented": stats[1],
            "no_consent": stats[2]
        }
        report["total_users"] += stats[0]
        report["consented"] += stats[1]
        report["no_consent"] += stats[2]

        # Users without consent (compliance risk)
        cur.execute("""
            SELECT id, email, full_name, country_code
            FROM users
            WHERE home_region = %s AND (consent_given = FALSE OR consent_given IS NULL)
        """, (region,))
        report["by_region"][region]["no_consent_users"] = cur.fetchall()

        # Consent by purpose
        cur.execute("""
            SELECT purpose, action, COUNT(*) as cnt
            FROM consent_log cl
            JOIN users u ON cl.user_id = u.id
            WHERE u.home_region = %s
            GROUP BY purpose, action
            ORDER BY purpose, action
        """, (region,))
        report["by_region"][region]["consent_details"] = cur.fetchall()

        conn.close()

    return report


# Generate and display
print("📋 GDPR CONSENT AUDIT REPORT")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

report = generate_consent_report()

# Summary
consent_rate = (report['consented'] / report['total_users'] * 100) if report['total_users'] > 0 else 0
print(f"\n📊 Overall Summary:")
print(f"   Total users: {report['total_users']}")
print(f"   With consent: {report['consented']} ({consent_rate:.0f}%)")
print(f"   Without consent: {report['no_consent']}")

# Per-region details
for region, data in report["by_region"].items():
    print(f"\n📦 {region.upper()}:")
    print(f"   Users: {data['total']} | Consented: {data['consented']} | No consent: {data['no_consent']}")

    if data["no_consent_users"]:
        print(f"   ⚠️  Users requiring attention:")
        for u in data["no_consent_users"]:
            print(f"      - {u[2]} ({u[1]}) — country: {u[3]}")

    if data["consent_details"]:
        print(f"   Consent breakdown:")
        for purpose, action, count in data["consent_details"]:
            icon = "✅" if action == "granted" else "❌"
            print(f"      {icon} {purpose}: {action} ({count})")

## 4. Erasure Request Tracking

If your organization has received GDPR erasure requests (Article 17), you need to track them and prove they were handled within the 30-day deadline:

In [ ]:
# ── Audit Tool 4: Erasure Request Status Report ───────────

def generate_erasure_report():
    """Generates a report of all erasure requests across regions."""
    all_requests = []

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, user_id, user_email, reason, status,
                   requested_at, completed_at, completed_by
            FROM erasure_requests
            ORDER BY requested_at DESC
        """)

        for row in cur.fetchall():
            days_elapsed = None
            if row[5]:  # requested_at
                if row[6]:  # completed_at
                    days_elapsed = (row[6] - row[5]).days
                else:
                    days_elapsed = (datetime.now() - row[5]).days

            all_requests.append({
                "region": region,
                "id": row[0],
                "user_email": row[2],
                "reason": row[3],
                "status": row[4],
                "requested_at": row[5],
                "completed_at": row[6],
                "days_elapsed": days_elapsed,
                "compliant": days_elapsed is not None and days_elapsed <= 30
            })
        conn.close()

    return all_requests


print("📋 ERASURE REQUEST STATUS REPORT")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

requests = generate_erasure_report()

if requests:
    table_data = []
    for r in requests:
        status_icon = {
            "completed": "✅",
            "processing": "🔄",
            "pending": "⏳",
            "denied": "❌"
        }.get(r["status"], "❓")

        deadline_status = ""
        if r["days_elapsed"] is not None:
            if r["status"] == "completed":
                deadline_status = f"{r['days_elapsed']}d (OK)" if r["compliant"] else f"{r['days_elapsed']}d (OVERDUE!)"
            else:
                remaining = 30 - r["days_elapsed"]
                deadline_status = f"{remaining}d left" if remaining > 0 else "OVERDUE!"

        table_data.append([
            r["region"],
            f"#{r['id']}",
            r["user_email"],
            f"{status_icon} {r['status']}",
            deadline_status
        ])

    print(tabulate(
        table_data,
        headers=["Region", "Request", "User Email", "Status", "30-Day Deadline"],
        tablefmt="grid"
    ))
else:
    print("\nNo erasure requests found.")
    print("(Run Notebook 3 first to generate erasure requests)")

## 5. Data Movement Audit Trail

Every time data moves between regions, it should be logged. This is critical for proving compliance during a DPA audit:

In [ ]:
# ── Audit Tool 5: Data Movement Log ───────────────────────

def generate_movement_report():
    """Shows all recorded data movements between regions."""
    movements = []

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT drl.user_id, drl.action, drl.source_region,
                   drl.target_region, drl.table_name, drl.reason,
                   drl.created_at
            FROM data_residency_log drl
            ORDER BY drl.created_at DESC
            LIMIT 20
        """)

        for row in cur.fetchall():
            movements.append({
                "logged_in": region,
                "user_id": row[0],
                "action": row[1],
                "source": row[2],
                "target": row[3] or "N/A",
                "table": row[4],
                "reason": row[5],
                "timestamp": row[6]
            })
        conn.close()

    # Sort by timestamp
    movements.sort(key=lambda x: x["timestamp"] if x["timestamp"] else datetime.min, reverse=True)
    return movements


print("📋 DATA MOVEMENT AUDIT TRAIL")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

movements = generate_movement_report()

if movements:
    action_icons = {
        "write": "✏️",
        "replicate": "🔄",
        "delete": "🗑️",
        "access": "👁️"
    }

    table_data = []
    for m in movements[:15]:  # Show most recent 15
        icon = action_icons.get(m["action"], "❓")
        table_data.append([
            str(m["timestamp"])[:19] if m["timestamp"] else "N/A",
            f"{icon} {m['action']}",
            f"User {m['user_id']}",
            m["source"],
            m["target"],
            m["table"],
        ])

    print(tabulate(
        table_data,
        headers=["Timestamp", "Action", "User", "Source", "Target", "Table"],
        tablefmt="grid"
    ))

    print(f"\n📊 Summary:")
    action_counts = {}
    for m in movements:
        action_counts[m["action"]] = action_counts.get(m["action"], 0) + 1
    for action, count in sorted(action_counts.items()):
        icon = action_icons.get(action, "❓")
        print(f"   {icon} {action}: {count} event(s)")
else:
    print("\nNo data movements recorded yet.")
    print("(Run Notebooks 1-3 to generate movement data)")

## 6. Full GDPR Compliance Dashboard

Let's combine everything into a single compliance dashboard — the kind of report you'd present to a Data Protection Officer (DPO) or a regulatory audit:

In [ ]:
# ── Full GDPR Compliance Dashboard ─────────────────────────

def generate_compliance_dashboard():
    """Generates a comprehensive GDPR compliance report."""
    print("╔" + "═" * 58 + "╗")
    print("║" + "  GDPR COMPLIANCE DASHBOARD".center(58) + "║")
    print("║" + f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(58) + "║")
    print("╚" + "═" * 58 + "╝")

    scores = []

    # ── Check 1: Data Residency ─────────────────────────────
    print("\n1️⃣  DATA RESIDENCY CHECK")
    violations, _ = detect_violations()
    if not violations:
        print("   ✅ PASS — All data is in the correct region")
        scores.append(100)
    else:
        print(f"   ❌ FAIL — {len(violations)} violation(s) detected")
        scores.append(0)

    # ── Check 2: Consent Coverage ──────────────────────────
    print("\n2️⃣  CONSENT COVERAGE CHECK")
    consent_report = generate_consent_report()
    consent_rate = (consent_report["consented"] / consent_report["total_users"] * 100) if consent_report["total_users"] > 0 else 0
    if consent_rate >= 95:
        print(f"   ✅ PASS — {consent_rate:.0f}% of users have valid consent")
        scores.append(100)
    elif consent_rate >= 80:
        print(f"   ⚠️  WARNING — {consent_rate:.0f}% consent rate (target: 95%+)")
        scores.append(70)
    else:
        print(f"   ❌ FAIL — Only {consent_rate:.0f}% consent rate")
        scores.append(30)

    # ── Check 3: Erasure SLA ───────────────────────────────
    print("\n3️⃣  ERASURE REQUEST SLA CHECK (30-day deadline)")
    erasure_requests = generate_erasure_report()
    overdue = [r for r in erasure_requests if r["status"] not in ["completed", "denied"] and r["days_elapsed"] and r["days_elapsed"] > 30]
    if not erasure_requests:
        print("   ℹ️  No erasure requests to evaluate")
        scores.append(100)
    elif not overdue:
        print(f"   ✅ PASS — All {len(erasure_requests)} request(s) within SLA")
        scores.append(100)
    else:
        print(f"   ❌ FAIL — {len(overdue)} request(s) overdue!")
        scores.append(0)

    # ── Check 4: Audit Trail Completeness ──────────────────
    print("\n4️⃣  AUDIT TRAIL CHECK")
    movements = generate_movement_report()
    has_write_logs = any(m["action"] == "write" for m in movements)
    has_delete_logs = any(m["action"] == "delete" for m in movements)

    audit_items = []
    if has_write_logs:
        audit_items.append("write operations logged")
    if has_delete_logs:
        audit_items.append("delete operations logged")

    if movements:
        print(f"   ✅ PASS — {len(movements)} event(s) in audit log")
        if audit_items:
            print(f"      Includes: {', '.join(audit_items)}")
        scores.append(100)
    else:
        print("   ⚠️  WARNING — No audit trail found")
        scores.append(50)

    # ── Overall Score ──────────────────────────────────────
    overall = sum(scores) / len(scores) if scores else 0
    grade = "A" if overall >= 90 else "B" if overall >= 80 else "C" if overall >= 70 else "D" if overall >= 60 else "F"

    print("\n" + "=" * 60)
    print(f"📊 OVERALL COMPLIANCE SCORE: {overall:.0f}/100 (Grade: {grade})")
    print("=" * 60)

    if overall >= 90:
        print("🎉 Excellent! Your system is well-prepared for a GDPR audit.")
    elif overall >= 70:
        print("⚠️  Good, but there are areas that need attention.")
    else:
        print("🚨 Significant compliance gaps detected. Immediate action required.")


# Run the dashboard
generate_compliance_dashboard()

## 7. Why Microsoft Builds Compliance Tools Into Azure

### Azure's Compliance Portfolio

| Azure Service | What It Does | Mapped to Our Lab |
|--------------|-------------|-------------------|
| **Microsoft Purview** | Auto-discovers and classifies PII across all services | Our `generate_residency_map()` |
| **Compliance Manager** | Scores your GDPR readiness, suggests improvements | Our `generate_compliance_dashboard()` |
| **Azure Policy** | Enforces rules (e.g., "no resources outside EU") | Our `detect_violations()` |
| **Azure Monitor** | Tracks all data access and movement | Our `data_residency_log` table |
| **Azure Key Vault** | Manages encryption keys per region | Not covered (encryption lab) |

### The Business Value

Enterprise customers pay premium prices for Azure because:
1. **Built-in compliance** saves months of engineering effort
2. **Shared responsibility** — Azure handles infrastructure compliance, you handle application compliance
3. **Certifications** — Azure is pre-certified for GDPR, ISO 27001, SOC 2, etc.
4. **Audit support** — Azure generates compliance reports automatically

The audit tools we built in this notebook are simplified versions of what Azure provides out of the box. In production, you'd use Azure's built-in tools plus your own application-level auditing.

## 🎯 Key Takeaways

1. **Data sovereignty** = data is subject to the laws where it's stored
2. **GDPR Article 30** requires a Record of Processing Activities (ROPA)
3. **Residency maps** show where each user's data physically lives
4. **Violation detection** catches data stored in the wrong region
5. **Consent audits** prove users agreed to data processing
6. **Erasure tracking** ensures the 30-day deadline is met
7. **Audit trails** prove compliance — log every data movement
8. **Azure provides built-in tools** (Purview, Compliance Manager, Policy) that do this at scale

## 🏁 Lab Complete!

You've now built a complete GDPR compliance system using the Azure Paired Regions pattern:

| Notebook | What You Built |
|----------|---------------|
| 1. Data Residency | Geo-routing writes to the correct region |
| 2. Cross-Region Replication | Async replication with failover |
| 3. Right to Erasure | Cascading deletes across regions |
| 4. Data Sovereignty Audit | Compliance reports and violation detection |

### Clean Up

```bash
cd enterprise-patterns/gdpr-paired-regions
docker-compose down -v  # Stops containers and removes data volumes
```